In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import s3fs
import umap
from dotenv import load_dotenv
from qdrant_client import QdrantClient

load_dotenv(override=True)

fs = s3fs.S3FileSystem(
    endpoint_url="https://minio.lab.sspcloud.fr",
    client_kwargs={"region_name": "us-east-1"},
)

qdrant_client = QdrantClient(
    url=os.environ.get("QDRANT_API_URL", "http://qdrant:6333"),
    api_key=os.environ.get("QDRANT_API_KEY", None),
    timeout=120,
)

HEURES_CLIP = 600

/home/onyxia/work/accords_heures_supp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_76547/1349662193.py:19: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(


In [2]:
def fetch_all(collection: str) -> pd.DataFrame:
    """Scroll the whole collection (vectors + payload) into a DataFrame."""
    points = []
    offset = None
    while True:
        res, offset = qdrant_client.scroll(
            collection_name=collection,
            limit=5000,
            offset=offset,
            with_vectors=True,
            with_payload=True,
        )
        points.extend(res)
        if offset is None:
            break

    records = []
    for p in points:
        payload = p.payload or {}
        meta = payload.get("metadata", {}) or {}
        rec = {"id": str(p.id), "page_content": payload.get("page_content")}
        rec.update(meta)
        records.append(rec)

    df = pd.DataFrame(records)
    df.attrs["vectors"] = np.asarray([p.vector for p in points], dtype=np.float32)
    print(f"Fetched {len(df)} points from '{collection}'.")
    return df

In [3]:
def run_umap(vectors: np.ndarray, n_neighbors: int, min_dist: float, metric: str, random_state: int) -> np.ndarray:
    print(
        f"Running UMAP (n={len(vectors)}, n_neighbors={n_neighbors}, min_dist={min_dist}, metric={metric})..."
    )
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric=metric,
        random_state=random_state,
    )
    return reducer.fit_transform(vectors)

In [4]:
df = fetch_all("population_concernee_contingent")

Fetched 14560 points from 'population_concernee_contingent'.


In [16]:
df["embeddings"] = list(df.attrs["vectors"])

In [17]:
df_2023 = df[pd.col("anneeSignature") == 2023]

In [19]:
df_2023.shape

(2177, 15)

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14560 entries, 0 to 14559
Data columns (total 15 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   id                                 14560 non-null  str    
 1   page_content                       14560 non-null  str    
 2   reference                          14560 non-null  str    
 3   anneeSignature                     14560 non-null  int64  
 4   n_chunks                           14560 non-null  int64  
 5   presence_contingent_conventionnel  14560 non-null  bool   
 6   nombre_heures_fixe                 14560 non-null  int64  
 7   type_substitution                  14560 non-null  str    
 8   periode                            14560 non-null  str    
 9   nombre_heures_avant_accord         3220 non-null   float64
 10  citation_exacte                    14560 non-null  str    
 11  justification_ou_reserve           14559 non-null  str    
 12  t

In [26]:
df.attrs.clear()

In [27]:
df.to_parquet("embeddings.parquet")

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14560 entries, 0 to 14559
Data columns (total 15 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   id                                 14560 non-null  str    
 1   page_content                       14560 non-null  str    
 2   reference                          14560 non-null  str    
 3   anneeSignature                     14560 non-null  int64  
 4   n_chunks                           14560 non-null  int64  
 5   presence_contingent_conventionnel  14560 non-null  bool   
 6   nombre_heures_fixe                 14560 non-null  int64  
 7   type_substitution                  14560 non-null  str    
 8   periode                            14560 non-null  str    
 9   nombre_heures_avant_accord         3220 non-null   float64
 10  citation_exacte                    14560 non-null  str    
 11  justification_ou_reserve           14559 non-null  str    
 12  t

In [33]:
embeddings_2023 = np.vstack(df_2023["embeddings"].to_list())

In [38]:
umap_proj = run_umap(
    vectors=embeddings_2023,
    n_neighbors=100,
    min_dist=0.5,
    metric="cosine",
    random_state=42
)

Running UMAP (n=2177, n_neighbors=100, min_dist=0.5, metric=cosine)...


/home/onyxia/work/accords_heures_supp/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [41]:
df_2023["umap_1"] = umap_proj[:,0]
df_2023["umap_2"] = umap_proj[:,1]

In [46]:
import plotly.express as px


fig = px.scatter(
    data_frame=df_2023[pd.col("nombre_heures_fixe") <=600],
    x="umap_1",
    y="umap_2",
    color="nombre_heures_fixe",
    hover_name="page_content"
)

fig.update_layout(height=800, width=1200)